In [ ]:
# Install required libraries
!pip install transformers datasets accelerate torch -q



In [ ]:
# Import all required libraries
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

In [ ]:
# Custom dataset - Mix of Stories and Jokes
text_data = """
Once upon a time, in a small village nestled between two tall mountains, there lived a young boy named Leo.
Leo loved to explore the forests and rivers near his home, always curious about the world around him.
One day, Leo discovered a mysterious cave hidden behind a waterfall.
Inside the cave, he found a glowing map that pointed to a hidden treasure deep in the forest.
Leo packed his bag with food and water and set off on the greatest adventure of his life.
Along the way, he met a talking fox who offered to guide him through the dangerous forest.
The fox was clever and wise, and together they solved every puzzle and crossed every obstacle.
After three days of adventure, Leo finally found the treasure chest buried under an ancient oak tree.
Inside the chest was not gold or jewels, but hundreds of books filled with magical stories.
Leo smiled and realized that the real treasure was the adventure and knowledge he had gained.

Why don't scientists trust atoms? Because they make up everything!
A man walked into a library and asked for books about paranoia. The librarian whispered, "They're right behind you!"
Why did the scarecrow win an award? Because he was outstanding in his field!
I told my wife she was drawing her eyebrows too high. She looked surprised.
Why can't you give Elsa a balloon? Because she will let it go!

Once there was a little girl named Mia who lived in a house full of books and dreams.
Mia wanted to become a great writer and write stories that would make people laugh and cry.
Every night, she would sit by her window and write in her little red notebook.
She wrote about dragons, stars, magical gardens and friendly monsters.
One day, a famous publisher read one of her stories and was completely amazed.
Mia's book was published and became the most loved story in the entire kingdom.
Children everywhere read her book and found joy, laughter and hope in her words.
Mia proved that dreams do come true if you work hard and never give up.

What do you call a fish without eyes? A fsh!
Why did the bicycle fall over? Because it was two tired!
A boy asked his father, "Dad, can you tell me what a solar eclipse is?" Dad replied, "No sun."
Why do cows wear bells? Because their horns don't work!
What do elves learn in school? The elf-abet!

There was once a kind old man who lived alone in a cottage by the sea.
Every morning he would walk to the shore and watch the waves crash against the rocks.
One stormy night, he heard a faint cry coming from the water.
He rushed out and found a small dolphin stranded on the beach.
With great effort, the old man guided the dolphin back into the ocean.
The next morning, the dolphin returned with a hundred others, dancing in the waves as if saying thank you.
The old man laughed and waved, his heart full of warmth and wonder.
From that day on, the dolphins visited him every morning without fail.
The old man never felt lonely again, for the sea had given him the most faithful of friends.

Why did the student eat his homework? Because the teacher told him it was a piece of cake!
What did one wall say to the other wall? I'll meet you at the corner!
Why did the golfer bring extra pants? In case he got a hole in one!
A ghost walks into a bar. The bartender says, "Sorry, we don't serve spirits here."
Why did the tomato turn red? Because it saw the salad dressing!
"""

# Save the dataset to a text file
with open("dataset.txt", "w") as f:
    f.write(text_data)

print("✅ Stories + Jokes dataset created and saved as dataset.txt!")

✅ Stories + Jokes dataset created and saved as dataset.txt!


In [ ]:
# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 doesn't have a padding token by default, so we set it
tokenizer.pad_token = tokenizer.eos_token

# Read the dataset file
with open("dataset.txt", "r") as f:
    text = f.read()

# Tokenize the text
def tokenize_data(text, tokenizer, max_length=512):
    # Split text into chunks
    lines = text.strip().split("\n")
    lines = [line.strip() for line in lines if line.strip()]

    # Tokenize each line
    tokenized = tokenizer(
        lines,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized

tokenized_data = tokenize_data(text, tokenizer)

print("✅ Tokenization complete!")
print(f"📊 Total samples tokenized: {len(tokenized_data['input_ids'])}")
print(f"🔢 Sample token IDs (first line): {tokenized_data['input_ids'][0][:10]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Tokenization complete!
📊 Total samples tokenized: 42
🔢 Sample token IDs (first line): tensor([ 7454,  2402,   257,   640,    11,   287,   257,  1402,  7404, 16343])...


In [ ]:
# Load GPT-2 pre-trained model
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("✅ GPT-2 Model loaded successfully!")
print(f"🧠 Model parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ GPT-2 Model loaded successfully!
🧠 Model parameters: 124,439,808


In [ ]:
# Prepare dataset in the format Trainer expects
from datasets import Dataset
import torch

# Read and split text into lines
with open("dataset.txt", "r") as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

# Create a Hugging Face Dataset
raw_dataset = Dataset.from_dict({"text": lines})

# Tokenize the dataset properly for training
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Apply tokenization
tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("✅ Dataset prepared for training!")
print(f"📊 Total training samples: {len(tokenized_dataset)}")

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

✅ Dataset prepared for training!
📊 Total training samples: 42


In [ ]:
# Setup Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT-2 uses causal language modeling, not masked
)

print("✅ Data Collator ready!")

✅ Data Collator ready!


In [ ]:
from transformers import TrainingArguments, Trainer

# Set up Training Arguments (Fully Updated)
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=5e-5,
    warmup_steps=20,
    logging_dir="./logs",
    report_to="none",
)

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("✅ Trainer is ready!")

# Start Training
print("🚀 Starting fine-tuning... please wait!")
trainer.train()
print("🎉 Fine-tuning complete!")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer is ready!
🚀 Starting fine-tuning... please wait!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.841769
20,3.774247
30,2.715344
40,2.759848
50,1.991538
60,1.909593
70,1.639377
80,1.315406
90,1.003118
100,1.094514


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Fine-tuning complete!


In [ ]:
# Save the fine-tuned model and tokenizer
model.save_pretrained("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("✅ Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved successfully!


In [ ]:
# Load the fine-tuned model for text generation
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer
)

print("✅ Generator ready!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Generator ready!


In [ ]:
# Generate Text! 🎉
prompts = [
    "Once upon a time",
    "Why did the",
    "There was a girl named",
]

for prompt in prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    print("-" * 50)
    result = generator(
        prompt,
        max_length=100,
        num_return_sequences=1,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        do_sample=True,
    )
    print(f"🤖 Generated: {result[0]['generated_text']}")
    print("=" * 50)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'max_length', 'top_k', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Prompt: 'Once upon a time'
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Generated: Once upon a time, in a small village nestled between two tall mountains, there lived a young boy named Leo. The boy loved to explore the forests and rivers near his home, always curious about the world around him. He never saw the world that his home had become. But over the years, he became convinced that the world was full of people who wanted to know more than him. Now, hundreds of years later, Leo has discovered hundreds of thousands of stories that make up the entire story. The adventure will give you the full story of Leo's life and work, as you work to bring the world together. The adventure begins with the adventure and the friends you make, and will make your days full of adventure and laughter. Enjoy the adventure and make friends with the people you meet, and together you will make the world a better place. Ages 4 to 12, Leo will make your dreams come true and make your dreams come true every day.


Key Features

● Ages 4 to 12 will make your dreams come true an

[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Generated: Why did the bicycle fall over? Because it was two tired! The bicycle was on its way to the top! A big thank you went out to the bicycle's owner! The bicycle was very happy and wanted to give it to him again! The owner was completely amazed! The bicycle was completely free of dangerous situations! The owner was completely sure that the bicycle was right behind him! The owner was completely confident that the bicycle was the one to bring the most friends to his home! The owner was completely amazed! The owner was completely sure that the bicycle was the one to bring the most friends to his home! The owner was completely confident that the bicycle was the one to bring the most friends to his home! The owner was completely confident that the bicycle was the one to bring the most friends to his home! The bicycle was completely free of dangerous situations! The bicycle was completely free of dangerous situations! The owner was completely confident that the bicycle was the one to

In [6]:
# ============================================
# COMPLETE GPT-2 FINE-TUNING IN ONE CELL
# ============================================
import os
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

# ---- STEP 1: Load Tokenizer & Model ----
print("⏳ Loading GPT-2...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")
print("✅ GPT-2 Loaded!")

# ---- STEP 2: Create Dataset ----
print("⏳ Creating dataset...")
text_data = """
Once upon a time, in a small village nestled between two tall mountains, there lived a young boy named Leo.
Leo loved to explore the forests and rivers near his home, always curious about the world around him.
One day, Leo discovered a mysterious cave hidden behind a waterfall.
Inside the cave, he found a glowing map that pointed to a hidden treasure deep in the forest.
Leo packed his bag with food and water and set off on the greatest adventure of his life.
Along the way, he met a talking fox who offered to guide him through the dangerous forest.
The fox was clever and wise, and together they solved every puzzle and crossed every obstacle.
After three days of adventure, Leo finally found the treasure chest buried under an ancient oak tree.
Inside the chest was not gold or jewels, but hundreds of books filled with magical stories.
Leo smiled and realized that the real treasure was the adventure and knowledge he had gained.

Why don't scientists trust atoms? Because they make up everything!
A man walked into a library and asked for books about paranoia. The librarian whispered, they are right behind you!
Why did the scarecrow win an award? Because he was outstanding in his field!
I told my wife she was drawing her eyebrows too high. She looked surprised.
Why can't you give Elsa a balloon? Because she will let it go!

Once there was a little girl named Mia who lived in a house full of books and dreams.
Mia wanted to become a great writer and write stories that would make people laugh and cry.
Every night, she would sit by her window and write in her little red notebook.
She wrote about dragons, stars, magical gardens and friendly monsters.
One day, a famous publisher read one of her stories and was completely amazed.
Mia's book was published and became the most loved story in the entire kingdom.
Children everywhere read her book and found joy, laughter and hope in her words.
Mia proved that dreams do come true if you work hard and never give up.

What do you call a fish without eyes? A fsh!
Why did the bicycle fall over? Because it was two tired!
A boy asked his father, Dad, can you tell me what a solar eclipse is? Dad replied, No sun.
Why do cows wear bells? Because their horns do not work!
What do elves learn in school? The elf-abet!

There was once a kind old man who lived alone in a cottage by the sea.
Every morning he would walk to the shore and watch the waves crash against the rocks.
One stormy night, he heard a faint cry coming from the water.
He rushed out and found a small dolphin stranded on the beach.
With great effort, the old man guided the dolphin back into the ocean.
The next morning, the dolphin returned with a hundred others, dancing in the waves as if saying thank you.
The old man laughed and waved, his heart full of warmth and wonder.
From that day on, the dolphins visited him every morning without fail.
The old man never felt lonely again, for the sea had given him the most faithful of friends.

Why did the student eat his homework? Because the teacher told him it was a piece of cake!
What did one wall say to the other wall? I will meet you at the corner!
Why did the golfer bring extra pants? In case he got a hole in one!
A ghost walks into a bar. The bartender says, Sorry, we do not serve spirits here.
Why did the tomato turn red? Because it saw the salad dressing!
"""

with open("dataset.txt", "w") as f:
    f.write(text_data)
print("✅ Dataset created!")

# ---- STEP 3: Tokenize ----
print("⏳ Tokenizing...")
lines = [line.strip() for line in text_data.strip().split("\n") if line.strip()]
raw_dataset = Dataset.from_dict({"text": lines})

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
print("✅ Tokenization done!")

# ---- STEP 4: Training ----
print("⏳ Setting up trainer...")
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=5e-5,
    warmup_steps=20,
    logging_dir="logs",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🚀 Starting fine-tuning...")
trainer.train()
print("🎉 Fine-tuning complete!")

# ---- STEP 5: Save ----
model.save_pretrained("gpt2-finetuned")
tokenizer.save_pretrained("gpt2-finetuned")
print("✅ Model saved!")

# ---- STEP 6: Generate Text ----
print("\n🤖 Generating text...\n")

def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompts = [
    "Once upon a time",
    "Why did the",
    "There was a girl named",
]

for prompt in prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    print("-" * 50)
    print(f"🤖 Generated: {generate_text(prompt)}")
    print("=" * 50)

⏳ Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ GPT-2 Loaded!
⏳ Creating dataset...
✅ Dataset created!
⏳ Tokenizing...


Map:   0%|          | 0/42 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Fine-tuning complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!

🤖 Generating text...


📝 Prompt: 'Once upon a time'
--------------------------------------------------
🤖 Generated: Once upon a time in the forest, there lived an old man named Leo. The only child he had ever known and never met again: his father's book was lost on another day when it returned with great adventure! Now together they found treasure deep within; for their journey home three generations to come new learnings...

📝 Prompt: 'Why did the'
--------------------------------------------------
🤖 Generated: Why did the tomato turn red? Because it saw a picture of his favorite book! The one with Mr. JUMPKIN'S COOKY MOUTHOMPEE!! He was right on time!!! We thank you, Santa Claus for your outstanding effort to bring this wonderful story home every day since January 1st...


Be sure and give thanks again as we receive more stories from across America each morning so that they may have fewer sleepless nights than their normal ones....

📝 Prompt: 'There was a girl named'
